# 📊 Speaker Verification - Model Comparison and Evaluation

Comparative analysis between **X-Vector** (baseline) and **ECAPA-TDNN** (SOTA).

### We evaluate and compare:
1. **Verification Accuracy**: F1-Score, EER, Precision, Recall, ROC AUC.
2. **Computational Complexity**: Parameter counts.
3. **Inference Performance**: Latency per clip.
4. **Resource Utilization**: Peak GPU memory, disk file size.
5. **Robustness**: Score separation visualizations.

## 🛠️ Step 1: Environment Setup

In [ ]:
!pip install -q torchaudio soundfile librosa matplotlib seaborn scikit-learn pandas numpy tqdm

import os, re, sys, time, math, random, glob, gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score, roc_curve, auc
import soundfile as sf
import librosa
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import torchaudio.transforms as T

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available(): print(f'GPU: {torch.cuda.get_device_name(0)}')

## 🧠 Step 2: Define Both Model Architectures

In [ ]:
# --- X-VECTOR ---
class TDNNBlock(nn.Module):
    def __init__(self, inc, outc, ks, dil):
        super().__init__()
        self.conv = nn.Conv1d(inc, outc, ks, dilation=dil, padding=int(dil*(ks-1)/2))
        self.bn = nn.BatchNorm1d(outc)
    def forward(self, x): return F.relu(self.bn(self.conv(x)))

class StatsPooling(nn.Module):
    def forward(self, x): return torch.cat([x.mean(2), x.std(2, unbiased=False)], 1)

class XVectorModel(nn.Module):
    def __init__(self, input_dim=80, num_classes=10, embedding_dim=512):
        super().__init__()
        self.tdnn1=TDNNBlock(input_dim,512,5,1); self.tdnn2=TDNNBlock(512,512,3,2)
        self.tdnn3=TDNNBlock(512,512,3,3); self.tdnn4=TDNNBlock(512,512,1,1)
        self.tdnn5=TDNNBlock(512,1500,1,1); self.pool=StatsPooling()
        self.fc1=nn.Linear(3000,embedding_dim); self.bn1=nn.BatchNorm1d(embedding_dim)
        self.fc2=nn.Linear(embedding_dim,embedding_dim); self.bn2=nn.BatchNorm1d(embedding_dim)
        self.classifier=nn.Linear(embedding_dim,num_classes)
    def extract_embedding(self, x):
        x=self.tdnn1(x);x=self.tdnn2(x);x=self.tdnn3(x);x=self.tdnn4(x);x=self.tdnn5(x)
        return self.bn1(self.fc1(self.pool(x)))
    def forward(self, x):
        e=self.extract_embedding(x); return self.classifier(F.relu(self.bn2(self.fc2(F.relu(e)))))

# --- ECAPA-TDNN ---
class SEBlock(nn.Module):
    def __init__(self, ch, r=8):
        super().__init__()
        self.fc=nn.Sequential(nn.Linear(ch,ch//r),nn.ReLU(),nn.Linear(ch//r,ch),nn.Sigmoid())
    def forward(self,x): return x*self.fc(x.mean(2)).unsqueeze(2)

class SERes2NetBlock(nn.Module):
    def __init__(self,ch,ks,dil,scale=4):
        super().__init__()
        self.scale=scale
        self.conv1=nn.Conv1d(ch,ch,1);self.bn1=nn.BatchNorm1d(ch)
        w=ch//scale
        self.convs=nn.ModuleList([nn.Conv1d(w,w,ks,dilation=dil,padding=int(dil*(ks-1)/2)) for _ in range(scale-1)])
        self.bns=nn.ModuleList([nn.BatchNorm1d(w) for _ in range(scale-1)])
        self.conv3=nn.Conv1d(ch,ch,1);self.bn3=nn.BatchNorm1d(ch)
        self.se=SEBlock(ch)
    def forward(self,x):
        res=x;out=F.relu(self.bn1(self.conv1(x)));sp=torch.chunk(out,self.scale,1);ns=[sp[0]]
        for i in range(1,self.scale):
            s=sp[i]+ns[i-1] if i>1 else sp[i]
            ns.append(F.relu(self.bns[i-1](self.convs[i-1](s))))
        return F.relu(self.se(self.bn3(self.conv3(torch.cat(ns,1))))+res)

class AttentiveStatsPooling(nn.Module):
    def __init__(self,inc,att=128):
        super().__init__()
        self.c1=nn.Conv1d(inc,att,1);self.c2=nn.Conv1d(att,inc,1)
    def forward(self,x):
        w=F.softmax(self.c2(torch.tanh(self.c1(x))),2)
        mu=(x*w).sum(2);sg=torch.sqrt(torch.clamp((w*(x-mu.unsqueeze(2))**2).sum(2),min=1e-9))
        return torch.cat([mu,sg],1)

class ECAPATDNNModel(nn.Module):
    def __init__(self,input_dim=80,num_classes=10,embedding_dim=192):
        super().__init__()
        self.conv1=nn.Conv1d(input_dim,512,5,padding=2);self.bn1=nn.BatchNorm1d(512)
        self.l1=SERes2NetBlock(512,3,2);self.l2=SERes2NetBlock(512,3,3);self.l3=SERes2NetBlock(512,3,4)
        self.conv2=nn.Conv1d(2048,1536,1);self.bn2=nn.BatchNorm1d(1536)
        self.pool=AttentiveStatsPooling(1536)
        self.fc=nn.Linear(3072,embedding_dim);self.bnfc=nn.BatchNorm1d(embedding_dim)
        self.classifier=nn.Linear(embedding_dim,num_classes)
    def extract_embedding(self,x):
        x0=F.relu(self.bn1(self.conv1(x)));x1=self.l1(x0);x2=self.l2(x1);x3=self.l3(x2)
        return self.bnfc(self.fc(self.pool(F.relu(self.bn2(self.conv2(torch.cat([x0,x1,x2,x3],1)))))))
    def forward(self,x): return self.classifier(F.relu(self.extract_embedding(x)))

print('Model architectures defined.')

## 🔄 Step 3: Load Checkpoints and Auto-Discover Test Data

In [ ]:
xvector_path = '/kaggle/input/models/ahmedmessoudi/models-ecapaxvector/keras/default/1/xvector_final_model.pt'
ecapa_path = '/kaggle/input/models/ahmedmessoudi/models-ecapaxvector/keras/default/1/ecapa_tdnn_final_model.pt'


assert os.path.exists(xvector_path), f'Missing {xvector_path}! Run Notebook 1 first.'
assert os.path.exists(ecapa_path), f'Missing {ecapa_path}! Run Notebook 2 first.'

xv_ckpt = torch.load(xvector_path, map_location=device, weights_only=False)
ec_ckpt = torch.load(ecapa_path, map_location=device, weights_only=False)
print('[OK] Both checkpoints loaded.')

xv_model = XVectorModel(xv_ckpt['input_dim'], xv_ckpt['num_classes'], xv_ckpt['embedding_dim'])
xv_model.load_state_dict(xv_ckpt['model_state_dict'])
xv_model = xv_model.to(device).eval()

ec_model = ECAPATDNNModel(ec_ckpt['input_dim'], ec_ckpt['num_classes'], ec_ckpt['embedding_dim'])
ec_model.load_state_dict(ec_ckpt['model_state_dict'])
ec_model = ec_model.to(device).eval()

# ============================================================
# AUTO-DISCOVER DATASET PATHS
# ============================================================
INPUT_ROOT = '/kaggle/input'

print('=' * 60)
print('Discovering /kaggle/input/ structure')
print('=' * 60)
for ds in sorted(os.listdir(INPUT_ROOT)):
    ds_path = os.path.join(INPUT_ROOT, ds)
    if os.path.isdir(ds_path):
        print(f'\n  [DIR] {ds}/')
        for sub in sorted(os.listdir(ds_path))[:10]:
            sub_path = os.path.join(ds_path, sub)
            tag = '[DIR]' if os.path.isdir(sub_path) else '[FILE]'
            print(f'    {tag} {sub}')
            if os.path.isdir(sub_path):
                for sub2 in sorted(os.listdir(sub_path))[:5]:
                    sub2_path = os.path.join(sub_path, sub2)
                    tag2 = '[DIR]' if os.path.isdir(sub2_path) else '[FILE]'
                    print(f'      {tag2} {sub2}')

print(f'\n{"=" * 60}')
print('Searching for all audio files...')
print('=' * 60)
all_wav = glob.glob(os.path.join(INPUT_ROOT, '**', '*.wav'), recursive=True)
all_flac = glob.glob(os.path.join(INPUT_ROOT, '**', '*.flac'), recursive=True)
print(f'  .wav  files found: {len(all_wav)}')
print(f'  .flac files found: {len(all_flac)}')
all_audio = all_wav + all_flac
print(f'  Total audio files: {len(all_audio)}')

if len(all_audio) > 0:
    print('\n  Sample paths:')
    for p in all_audio[:10]:
        print(f'    {p}')

# ============================================================
# CLASSIFY: VoxCeleb vs MUSAN
# ============================================================
print(f'\n{"=" * 60}')
print('Classifying audio files (VoxCeleb vs MUSAN)')
print('=' * 60)

vox_files = []
noise_files = []

for f in all_audio:
    fp = f.replace('\\', '/')
    if re.search(r'/id\d{3,}/', fp):
        vox_files.append(f)
    elif 'noise' in fp.lower() and 'musan' in fp.lower():
        noise_files.append(f)

# Fallback: broader match if primary regex found nothing
if len(vox_files) == 0:
    print('  [WARN] No VoxCeleb files found by id pattern. Trying broader match...')
    for f in all_audio:
        fp = f.replace('\\', '/').lower()
        if 'vox' in fp or 'celeb' in fp:
            vox_files.append(f)

if len(noise_files) == 0:
    print('  [WARN] No MUSAN noise files found. Trying broader match...')
    for f in all_audio:
        fp = f.replace('\\', '/').lower()
        if 'noise' in fp:
            noise_files.append(f)

print(f'  VoxCeleb audio files: {len(vox_files)}')
print(f'  MUSAN noise files:    {len(noise_files)}')

assert len(vox_files) > 0, (
    'ERROR: No VoxCeleb audio files found! '
    'Please check that the VoxCeleb dataset is correctly attached in Kaggle.'
)

# ============================================================
# BUILD SPEAKER DICTIONARY
# ============================================================
def get_spk(p):
    for x in p.replace('\\', '/').split('/'):
        if re.match(r'^id\d{3,}$', x): return x
    return 'unknown'

speaker_dict = {}
for f in vox_files:
    s = get_spk(f)
    if s != 'unknown': speaker_dict.setdefault(s, []).append(f)

# Fallback: use parent directory names if no idXXXXX pattern found
if len(speaker_dict) == 0:
    print('  [WARN] No idXXXXX folders found. Using parent directories as speaker labels.')
    for f in vox_files:
        parts = f.replace('\\', '/').split('/')
        if len(parts) >= 3:
            speaker_dict.setdefault(parts[-3], []).append(f)

print(f'  Total unique speakers: {len(speaker_dict)}')

# Scalable full/subset selection (None = use ALL speakers)
SELECT_SUBSET_SPEAKERS = None
if SELECT_SUBSET_SPEAKERS is not None and len(speaker_dict) > SELECT_SUBSET_SPEAKERS:
    sorted_spk = sorted(speaker_dict.items(), key=lambda x: len(x[1]), reverse=True)
    speaker_dict = dict(sorted_spk[:SELECT_SUBSET_SPEAKERS])
    print(f'  Using subset of {SELECT_SUBSET_SPEAKERS} speakers')

# ============================================================
# BUILD VALIDATION FILES
# ============================================================
val_files = []
for spk, files in speaker_dict.items():
    if len(files) >= 4:
        _, va = train_test_split(files, test_size=0.25, random_state=42)
        val_files.extend([(f, spk) for f in va])
    else:
        val_files.extend([(f, spk) for f in files])

print(f'  Validation files: {len(val_files)}')
assert len(val_files) > 0, 'ERROR: No validation files found! Check dataset attachment.'

# ============================================================
# GENERATE BENCHMARK PAIRS
# ============================================================
def gen_pairs(fl, n=1500, seed=7777):
    random.seed(seed); pairs = []; d = {}
    for f, s in fl: d.setdefault(s, []).append(f)
    spks = list(d.keys())
    assert len(spks) > 0, 'ERROR: No speakers with files found for pair generation!'
    att = 0
    while len(pairs) < n // 2 and att < 15000:
        att += 1; s = random.choice(spks)
        if len(d[s]) >= 2: a, b = random.sample(d[s], 2); pairs.append((a, b, 1))
    att = 0
    while len(pairs) < n and att < 15000:
        att += 1; s1, s2 = random.sample(spks, 2)
        pairs.append((random.choice(d[s1]), random.choice(d[s2]), 0))
    return pairs

comp_pairs = gen_pairs(val_files, 1500, 8888)
print(f'\nBenchmark pairs generated: {len(comp_pairs)}')
print(f'  Positive pairs: {sum(1 for _,_,l in comp_pairs if l==1)}')
print(f'  Negative pairs: {sum(1 for _,_,l in comp_pairs if l==0)}')

## ⚡ Step 4: Hardware Efficiency Benchmark

In [ ]:
def count_params(m): return sum(p.numel() for p in m.parameters() if p.requires_grad)

xv_params = count_params(xv_model)
ec_params = count_params(ec_model)
xv_sz = os.path.getsize(xvector_path)/(1024*1024)
ec_sz = os.path.getsize(ecapa_path)/(1024*1024)

mel_transform = T.MelSpectrogram(sample_rate=16000, n_fft=400, win_length=400, hop_length=160, n_mels=80).to(device)

def bench(mdl, pairs, nw=30, ne=150):
    ufs = list(set([p[0] for p in pairs]+[p[1] for p in pairs]))[:ne]
    mdl.eval()
    dummy = torch.randn(1,80,300).to(device)
    for _ in range(nw):
        with torch.no_grad(): mdl.extract_embedding(dummy)
    if torch.cuda.is_available(): torch.cuda.synchronize()
    t0 = time.perf_counter()
    with torch.no_grad():
        for fp in ufs:
            try: w,_=librosa.load(fp,sr=16000)
            except: w=np.zeros(48000,dtype=np.float32)
            if len(w)<48000: w=np.pad(w,(0,48000-len(w)))
            else: w=w[:48000]
            wt=torch.tensor(w,dtype=torch.float32).unsqueeze(0).to(device)
            lm=torch.log(mel_transform(wt)+1e-6); lm=lm-lm.mean()
            mdl.extract_embedding(lm)
    if torch.cuda.is_available(): torch.cuda.synchronize()
    lat = (time.perf_counter()-t0)*1000/len(ufs)
    if torch.cuda.is_available(): torch.cuda.reset_peak_memory_stats()
    with torch.no_grad(): mdl.extract_embedding(torch.randn(32,80,300).to(device))
    pmem = torch.cuda.max_memory_allocated()/1e6 if torch.cuda.is_available() else 0
    return lat, pmem

print('Benchmarking X-Vector...'); xv_lat, xv_mem = bench(xv_model, comp_pairs)
print('Benchmarking ECAPA-TDNN...'); ec_lat, ec_mem = bench(ec_model, comp_pairs)

edf = pd.DataFrame({
    'Metric': ['Parameters','File Size','Latency/clip','Peak GPU Mem (B=32)'],
    'X-Vector': [f'{xv_params:,}',f'{xv_sz:.1f} MB',f'{xv_lat:.1f} ms',f'{xv_mem:.0f} MB' if xv_mem else 'N/A'],
    'ECAPA-TDNN': [f'{ec_params:,}',f'{ec_sz:.1f} MB',f'{ec_lat:.1f} ms',f'{ec_mem:.0f} MB' if ec_mem else 'N/A']
})
print('\n' + edf.to_string(index=False))

## 🎯 Step 5: Verification Accuracy Benchmark

In [ ]:
def run_eval(mdl, pairs, th, dev):
    mdl.eval(); sims=[]; labs=[]; cache={}
    for a,b,lb in tqdm(pairs, desc='Evaluating'):
        for f in [a,b]:
            if f not in cache:
                try: w,_=librosa.load(f,sr=16000)
                except: w=np.zeros(48000,dtype=np.float32)
                if len(w)<48000: w=np.pad(w,(0,48000-len(w)))
                else: w=w[:48000]
                wt=torch.tensor(w,dtype=torch.float32).unsqueeze(0).to(dev)
                with torch.no_grad():
                    lm=torch.log(mel_transform(wt)+1e-6); lm=lm-lm.mean()
                    e=mdl.extract_embedding(lm)
                    cache[f]=F.normalize(e,p=2,dim=1).cpu().numpy()[0]
        sims.append(np.dot(cache[a],cache[b])); labs.append(lb)
    del cache; gc.collect(); torch.cuda.empty_cache()
    sims,labs = np.array(sims), np.array(labs)
    preds=(sims>=th).astype(int)
    fpr,tpr,_=roc_curve(labs,sims); fnr=1-tpr
    return {'f1':f1_score(labs,preds),'precision':precision_score(labs,preds,zero_division=0),
            'recall':recall_score(labs,preds,zero_division=0),'accuracy':accuracy_score(labs,preds),
            'eer':fpr[np.nanargmin(np.abs(fpr-fnr))],'sims':sims,'labels':labs,'fpr':fpr,'tpr':tpr}

print('Evaluating X-Vector...')
xv_r = run_eval(xv_model, comp_pairs, xv_ckpt['optimal_threshold'], device)
print('Evaluating ECAPA-TDNN...')
ec_r = run_eval(ec_model, comp_pairs, ec_ckpt['optimal_threshold'], device)

adf = pd.DataFrame({
    'Metric': ['F1-Score','EER','Accuracy','Precision','Recall'],
    'X-Vector': [f'{xv_r["f1"]:.4f}',f'{xv_r["eer"]:.4f}',f'{xv_r["accuracy"]:.4f}',f'{xv_r["precision"]:.4f}',f'{xv_r["recall"]:.4f}'],
    'ECAPA-TDNN': [f'{ec_r["f1"]:.4f}',f'{ec_r["eer"]:.4f}',f'{ec_r["accuracy"]:.4f}',f'{ec_r["precision"]:.4f}',f'{ec_r["recall"]:.4f}']
})
print('\n' + adf.to_string(index=False))

## 📊 Step 6: Comparative Visualizations

In [ ]:
os.makedirs('results', exist_ok=True)
fig, axes = plt.subplots(2, 2, figsize=(18, 12))

# 1. Metrics bar chart
metrics = ['F1','Accuracy','Precision','Recall']
xvs = [xv_r['f1'],xv_r['accuracy'],xv_r['precision'],xv_r['recall']]
ecs = [ec_r['f1'],ec_r['accuracy'],ec_r['precision'],ec_r['recall']]
x = np.arange(len(metrics)); w=0.35
axes[0,0].bar(x-w/2, xvs, w, label='X-Vector', color='#1f77b4')
axes[0,0].bar(x+w/2, ecs, w, label='ECAPA-TDNN', color='#2ca02c')
axes[0,0].set_xticks(x, metrics); axes[0,0].set_ylim(0,1.05)
axes[0,0].set_title('Verification Accuracy', fontweight='bold'); axes[0,0].legend(); axes[0,0].grid(alpha=0.3, axis='y')

# 2. EER comparison
axes[0,1].bar([-0.2],[xv_r['eer']],0.3,label='X-Vector',color='#1f77b4')
axes[0,1].bar([0.2],[ec_r['eer']],0.3,label='ECAPA-TDNN',color='#2ca02c')
axes[0,1].set_title('EER (Lower = Better)', fontweight='bold'); axes[0,1].legend(); axes[0,1].grid(alpha=0.3, axis='y')

# 3. ROC curves
axes[1,0].plot(xv_r['fpr'],xv_r['tpr'],'#1f77b4',lw=2.5,label=f'X-Vec AUC={auc(xv_r["fpr"],xv_r["tpr"]):.4f}')
axes[1,0].plot(ec_r['fpr'],ec_r['tpr'],'#2ca02c',lw=2.5,label=f'ECAPA AUC={auc(ec_r["fpr"],ec_r["tpr"]):.4f}')
axes[1,0].plot([0,1],[0,1],'--',color='grey')
axes[1,0].set_title('Combined ROC', fontweight='bold'); axes[1,0].legend(loc='lower right'); axes[1,0].grid(alpha=0.3)

# 4. DET Curves (Log-scale FAR vs FRR / FPR vs FNR)
fnr_xv = 1.0 - xv_r['tpr']
fnr_ec = 1.0 - ec_r['tpr']
axes[1,1].plot(xv_r['fpr'], fnr_xv, '#1f77b4', lw=2.5, label='X-Vector')
axes[1,1].plot(ec_r['fpr'], fnr_ec, '#2ca02c', lw=2.5, label='ECAPA-TDNN')
axes[1,1].set_xscale('log'); axes[1,1].set_yscale('log')
axes[1,1].set_xlim([1e-4, 1.0]); axes[1,1].set_ylim([1e-4, 1.0])
axes[1,1].set_xlabel('False Alarm Rate (FAR / FPR)', fontweight='bold')
axes[1,1].set_ylabel('False Reject Rate (FRR / FNR)', fontweight='bold')
axes[1,1].set_title('Detection Error Tradeoff (DET) Curves', fontweight='bold')
axes[1,1].legend(); axes[1,1].grid(True, which='both', alpha=0.3)

plt.tight_layout(); plt.savefig('results/model_comparison_report.png', dpi=150); plt.show()


## 🔊 Step 6.5: Comparative Noise Stress Test
We run comparative verification stress tests on **both models** under fixed noise levels (0dB, 10dB, 20dB) from the MUSAN noise dataset and evaluate how their F1-score, Equal Error Rate (EER), and Accuracy degrade under increasingly difficult noise environments.

In [ ]:
os.makedirs('results', exist_ok=True)
# noise_files already discovered in Step 3 above
print(f'MUSAN noise files available: {len(noise_files)}')

def eval_verif_noise(mdl, pairs, mt, dev, snr_val, threshold):
    mdl.eval()
    sims, labs = [], []
    cache = {}
    for a, b, lb in tqdm(pairs, desc=f'Evaluating {snr_val}dB'):
        for f in [a, b]:
            if f not in cache:
                try:
                    w, _ = librosa.load(f, sr=16000)
                except Exception:
                    w = np.zeros(48000, dtype=np.float32)
                ns = 48000
                if len(w) < ns: w = np.pad(w, (0, ns-len(w)))
                else: w = w[:ns]
                
                # Mix in random MUSAN background noise at exactly the fixed SNR
                if len(noise_files) > 0:
                    try:
                        nw, _ = librosa.load(random.choice(noise_files), sr=16000)
                        if len(nw) < len(w):
                            nw = np.tile(nw, math.ceil(len(w)/len(nw)))[:len(w)]
                        else:
                            nw = nw[:len(w)]
                        clean_p = np.mean(w**2) + 1e-8
                        noise_p = np.mean(nw**2) + 1e-8
                        scale = math.sqrt(clean_p / noise_p * 10**(-snr_val/10))
                        w = w + scale * nw
                        mx = np.max(np.abs(w))
                        if mx > 1.0: w = w / mx
                    except Exception:
                        pass
                wt = torch.tensor(w, dtype=torch.float32).unsqueeze(0).to(dev)
                with torch.no_grad():
                    m = mt(wt)
                    lm = torch.log(m+1e-6); lm = lm - lm.mean()
                    e = mdl.extract_embedding(lm)
                cache[f] = F.normalize(e, p=2, dim=1).cpu().numpy()[0]
        sims.append(np.dot(cache[a], cache[b]))
        labs.append(lb)
    del cache; gc.collect(); torch.cuda.empty_cache()
    sims, labs = np.array(sims), np.array(labs)
    preds = (sims >= threshold).astype(int)
    fpr, tpr, _ = roc_curve(labs, sims)
    fnr = 1 - tpr
    eer = fpr[np.nanargmin(np.abs(fpr - fnr))]
    return {'f1': f1_score(labs, preds), 'eer': eer, 'accuracy': accuracy_score(labs, preds)}

print('Evaluating both models under fixed noise conditions (0dB, 10dB, 20dB)...')
noise_results_xv = {}
noise_results_ec = {}
for snr in [20, 10, 0]:
    print(f'\n--- SNR = {snr} dB ---')
    noise_results_xv[snr] = eval_verif_noise(xv_model, comp_pairs, mel_transform, device, snr, xv_ckpt['optimal_threshold'])
    noise_results_ec[snr] = eval_verif_noise(ec_model, comp_pairs, mel_transform, device, snr, ec_ckpt['optimal_threshold'])

print(f'\n{"=" * 40} NOISE STRESS TEST COMPARISON {"=" * 40}')
data_noise = {
    'Environment': ['Clean Baseline', 'SNR 20 dB (Low)', 'SNR 10 dB (Medium)', 'SNR 0 dB (Heavy)'],
    'X-Vector F1': [f'{xv_r["f1"]:.4f}', f'{noise_results_xv[20]["f1"]:.4f}', f'{noise_results_xv[10]["f1"]:.4f}', f'{noise_results_xv[0]["f1"]:.4f}'],
    'ECAPA-TDNN F1': [f'{ec_r["f1"]:.4f}', f'{noise_results_ec[20]["f1"]:.4f}', f'{noise_results_ec[10]["f1"]:.4f}', f'{noise_results_ec[0]["f1"]:.4f}'],
    'X-Vector EER': [f'{xv_r["eer"]:.4f}', f'{noise_results_xv[20]["eer"]:.4f}', f'{noise_results_xv[10]["eer"]:.4f}', f'{noise_results_xv[0]["eer"]:.4f}'],
    'ECAPA-TDNN EER': [f'{ec_r["eer"]:.4f}', f'{noise_results_ec[20]["eer"]:.4f}', f'{noise_results_ec[10]["eer"]:.4f}', f'{noise_results_ec[0]["eer"]:.4f}']
}
df_noise = pd.DataFrame(data_noise)
print(df_noise.to_string(index=False))

# Plot comparative degradation curve
snr_levels = ['Clean', '20dB', '10dB', '0dB']
f1_xv_all = [xv_r['f1'], noise_results_xv[20]['f1'], noise_results_xv[10]['f1'], noise_results_xv[0]['f1']]
f1_ec_all = [ec_r['f1'], noise_results_ec[20]['f1'], noise_results_ec[10]['f1'], noise_results_ec[0]['f1']]

plt.figure(figsize=(10, 6))
plt.plot(snr_levels, f1_xv_all, marker='o', lw=2.5, ls='--', label='X-Vector (F1-Score)', color='#1f77b4')
plt.plot(snr_levels, f1_ec_all, marker='s', lw=2.5, label='ECAPA-TDNN (F1-Score)', color='#2ca02c')
plt.xlabel('Noise Condition / SNR', fontweight='bold')
plt.ylabel('F1-Score', fontweight='bold')
plt.title('Speaker Verification Degradation Under Noise Stress', fontweight='bold', fontsize=12)
plt.grid(True, alpha=0.3); plt.legend()
plt.savefig('results/noise_stress_degradation_comparison.png', dpi=150)
plt.show()


## 📝 Step 7: Summary and Interpretation

This final section generates a comprehensive summary report comparing both models across all evaluation dimensions.

In [ ]:
# ============================================================
# COMPREHENSIVE SUMMARY REPORT
# ============================================================
print('=' * 80)
print('           SPEAKER VERIFICATION - COMPREHENSIVE MODEL COMPARISON')
print('=' * 80)

print('\n' + '-' * 80)
print(' 1. HARDWARE EFFICIENCY')
print('-' * 80)
print(edf.to_string(index=False))

print('\n' + '-' * 80)
print(' 2. VERIFICATION ACCURACY (Clean Conditions)')
print('-' * 80)
print(adf.to_string(index=False))

print('\n' + '-' * 80)
print(' 3. NOISE ROBUSTNESS')
print('-' * 80)
print(df_noise.to_string(index=False))

# Determine winner per category
print('\n' + '-' * 80)
print(' 4. OVERALL ASSESSMENT')
print('-' * 80)

# Accuracy winner
if ec_r['f1'] > xv_r['f1']:
    acc_winner = 'ECAPA-TDNN'
    acc_delta = ec_r['f1'] - xv_r['f1']
elif xv_r['f1'] > ec_r['f1']:
    acc_winner = 'X-Vector'
    acc_delta = xv_r['f1'] - ec_r['f1']
else:
    acc_winner = 'Tie'
    acc_delta = 0.0

# EER winner (lower is better)
if ec_r['eer'] < xv_r['eer']:
    eer_winner = 'ECAPA-TDNN'
    eer_delta = xv_r['eer'] - ec_r['eer']
elif xv_r['eer'] < ec_r['eer']:
    eer_winner = 'X-Vector'
    eer_delta = ec_r['eer'] - xv_r['eer']
else:
    eer_winner = 'Tie'
    eer_delta = 0.0

# Param efficiency winner (fewer params = better)
param_winner = 'ECAPA-TDNN' if ec_params < xv_params else 'X-Vector'

# Noise robustness winner (higher F1 at 0dB)
xv_f1_0db = noise_results_xv[0]['f1']
ec_f1_0db = noise_results_ec[0]['f1']
noise_winner = 'ECAPA-TDNN' if ec_f1_0db > xv_f1_0db else 'X-Vector'

print(f'\n  Accuracy (F1-Score):     {acc_winner} wins' + (f' (+{acc_delta:.4f})' if acc_delta > 0 else ''))
print(f'  Error Rate (EER):        {eer_winner} wins' + (f' (-{eer_delta:.4f})' if eer_delta > 0 else ''))
print(f'  Parameter Efficiency:    {param_winner} ({min(xv_params, ec_params):,} vs {max(xv_params, ec_params):,})')
print(f'  Noise Robustness (0dB):  {noise_winner} (F1: {max(xv_f1_0db, ec_f1_0db):.4f} vs {min(xv_f1_0db, ec_f1_0db):.4f})')

print('\n' + '-' * 80)
print(' 5. KEY INSIGHTS')
print('-' * 80)
print('''
  - ECAPA-TDNN uses SE (Squeeze-and-Excitation) channel attention and multi-scale
    Res2Net blocks, enabling it to adaptively focus on the most speaker-discriminative
    frequency bands. This leads to higher verification accuracy.

  - X-Vector uses a simpler feed-forward TDNN architecture with statistics pooling.
    It is faster to train and has a more straightforward implementation.

  - Attentive Statistics Pooling (ECAPA-TDNN) captures weighted mean and standard
    deviation across time, providing richer utterance-level representations compared
    to the simple mean+std pooling in X-Vector.

  - Under heavy noise conditions (0dB SNR), both models degrade, but the model with
    attention mechanisms typically retains more discriminative power.

  - The DET curves on log-log scale reveal the operating characteristics at extreme
    false alarm / false reject rates, which are critical for real-world deployment.
''')

# Save summary report to text file
os.makedirs('results', exist_ok=True)
with open('results/comparison_summary.txt', 'w') as f:
    f.write('SPEAKER VERIFICATION - MODEL COMPARISON SUMMARY\n')
    f.write('=' * 60 + '\n\n')
    f.write('HARDWARE EFFICIENCY\n')
    f.write(edf.to_string(index=False) + '\n\n')
    f.write('VERIFICATION ACCURACY (Clean)\n')
    f.write(adf.to_string(index=False) + '\n\n')
    f.write('NOISE ROBUSTNESS\n')
    f.write(df_noise.to_string(index=False) + '\n\n')
    f.write(f'Accuracy Winner: {acc_winner}\n')
    f.write(f'EER Winner: {eer_winner}\n')
    f.write(f'Noise Robustness Winner: {noise_winner}\n')

print('\n[OK] Summary report saved to results/comparison_summary.txt')
print('=' * 80)
print('                          EVALUATION COMPLETE')
print('=' * 80)